In [16]:
%cd /storage/plzen4-ntis/home/jmatouse/experimenty/StyleTTS2_cs-modif/
%pwd

/auto/plzen4-ntis/home/jmatouse/experimenty/StyleTTS2_cs-modif


/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'/auto/plzen4-ntis/home/jmatouse/experimenty/StyleTTS2_cs-modif'

In [17]:
# load packages
import os
import sys
import time

import torch
import torch.nn.functional as F
import torchaudio
import yaml
from models import build_model, load_ASR_models, load_F0_models
from Modules.diffusion.sampler import ADPM2Sampler, DiffusionSampler, KarrasSchedule
from text_utils import TextCleaner
from utils import recursive_munch
from Utils.PLBERT.util import load_plbert

# import librosa
# from nltk.tokenize import word_tokenize

In [18]:
# Load processed training config
config = yaml.safe_load(open("Exps/KleIl/config2a.yml"))

# Load pretrained models
text_aligner = load_ASR_models(config["ASR_path"], config["ASR_config"])  # Text aligner
pitch_extractor = load_F0_models(config["F0_path"])  # F0 extractor
plbert = load_plbert(config["PLBERT_dir"])  # PLBERT

/auto/plzen4-ntis/home/jmatouse/experimenty/StyleTTS2_cs-modif/models.py:686: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  params = torch.load(model_path, map_location="cpu

Loading PL-BERT at Utils/PLBERT//step_8000000.t7 ...


In [19]:
# Build StyleTTS2 model
model = build_model(
    recursive_munch(config["model_params"]), text_aligner, pitch_extractor, plbert
)
_ = [model[key].eval() for key in model]  # Set model to eval mode

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [21]:
params_whole = torch.load("Exps/KleIl/stage2_pre-joint_00049.pth", map_location="cpu")

/scratch.ssd/jmatouse/job_8405813.pbs-m1.metacentrum.cz/ipykernel_142984/3423013751.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  params_whole = torch.load("Exps/KleIl

In [22]:
params_whole.keys()

dict_keys(['net', 'optimizer', 'iters', 'val_loss', 'epoch', 'sigma_data'])

In [23]:
params_whole["sigma_data"]

0.46833778888303573

In [24]:
params_whole["net"].keys()

dict_keys(['bert', 'bert_encoder', 'predictor', 'decoder', 'text_encoder', 'predictor_encoder', 'style_encoder', 'diffusion', 'text_aligner', 'pitch_extractor', 'mpd', 'msd', 'wd'])